In [1]:
import pandas as pd
df = pd.read_csv("../data/raw/bbri_historical.csv", index_col="Date", parse_dates=True)
df.head()

,Close,High,Low,Open,Volume
Date,,,,,
2020-01-02,2680.164062,2680.164062,2649.776650,2674.086547,45886302
2020-01-03,2686.241211,2698.396240,2668.008830,2686.241211,91189705
2020-01-06,2655.854004,2668.008870,2625.466594,2649.776489,48648450
2020-01-07,2674.086182,2680.163696,2661.931316,2680.163696,114344885
2020-01-08,2661.931152,2674.086017,2637.621260,2661.931152,188929583


In [2]:
# Lag features: harga penutupan 1, 2, 3, 5 hari sebelumnya
for lag in [1, 2, 3, 5]:
    df[f'close_lag_{lag}'] = df['Close'].shift(lag)

# Moving average: rata-rata harga 5 dan 10 hari terakhir
df['ma_5'] = df['Close'].rolling(window=5).mean()
df['ma_10'] = df['Close'].rolling(window=10).mean()

# Target: harga penutupan BESOK (geser -1, artinya "masa depan")
df['target'] = df['Close'].shift(-1)

df[['Close', 'close_lag_1', 'ma_5', 'ma_10', 'target']].head(15)

,Close,close_lag_1,ma_5,ma_10,target
Date,,,,,
2020-01-02,2680.164062,NaN,NaN,NaN,2686.241211
2020-01-03,2686.241211,2680.164062,NaN,NaN,2655.854004
2020-01-06,2655.854004,2686.241211,NaN,NaN,2674.086182
2020-01-07,2674.086182,2655.854004,NaN,NaN,2661.931152
2020-01-08,2661.931152,2674.086182,2671.655322,NaN,2674.086182
2020-01-09,2674.086182,2661.931152,2670.439746,NaN,2680.164062
2020-01-10,2680.164062,2674.086182,2669.224316,NaN,2740.938477
2020-01-13,2740.938477,2680.164062,2686.241211,NaN,2777.403320
2020-01-14,2777.403320,2740.938477,2706.904639,NaN,2783.480713


In [3]:
df_clean = df.dropna()
print(f"Sebelum: {len(df)} baris, sesudah dropna: {len(df_clean)} baris")

df_clean.to_csv("../data/processed/bbri_features.csv")

Sebelum: 1606 baris, sesudah dropna: 1596 baris


In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

feature_cols = ['close_lag_1', 'close_lag_2', 'close_lag_3', 'close_lag_5', 'ma_5', 'ma_10']
X = df_clean[feature_cols]
y = df_clean['target']

# Split KRONOLOGIS - bukan acak, karena ini time series
split_idx = int(len(df_clean) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train: {len(X_train)} baris ({X_train.index.min()} - {X_train.index.max()})")
print(f"Test: {len(X_test)} baris ({X_test.index.min()} - {X_test.index.max()})")

Train: 1276 baris (2020-01-15 00:00:00 - 2025-04-30 00:00:00)
Test: 320 baris (2025-05-02 00:00:00 - 2026-08-28 00:00:00)


In [10]:
import mlflow
import mlflow.sklearn
import joblib

mlflow.set_experiment("bbri-stock-forecast")

with mlflow.start_run(run_name="baseline_linear_regression"):
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    r2 = r2_score(y_test, y_pred)

    print(f"MAE: {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R²: {r2:.4f}")

    mlflow.log_param("model", "LinearRegression")
    mlflow.log_param("features", str(feature_cols))
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)
    mlflow.log_metric("naive_baseline_mae", naive_mae)
    mlflow.log_metric("naive_baseline_rmse", naive_rmse)
    mlflow.log_metric("naive_baseline_r2", naive_r2)

    mlflow.sklearn.log_model(model, "model")
    joblib.dump(model, "../models/bbri_price_model.joblib")

Exception: Run with UUID ffeeeca41974477ebc9525a3a7c2d2ce is already active. To start a new run, first end the current run with mlflow.end_run(). To start a nested run, call start_run with nested=True

In [8]:
naive_pred = X_test['close_lag_1']  # tebakan: besok = harga kemarin (hari ini)

naive_mae = mean_absolute_error(y_test, naive_pred)
naive_rmse = mean_squared_error(y_test, naive_pred) ** 0.5
naive_r2 = r2_score(y_test, naive_pred)

print("=== Baseline Naif (besok = hari ini) ===")
print(f"MAE: {naive_mae:.2f}")
print(f"RMSE: {naive_rmse:.2f}")
print(f"R²: {naive_r2:.4f}")

print("\n=== Model Linear Regression ===")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.4f}")

=== Baseline Naif (besok = hari ini) ===
MAE: 67.88
RMSE: 92.59
R²: 0.8962

=== Model Linear Regression ===
MAE: 56.35
RMSE: 74.19
R²: 0.9334
